# Fake Jobs – FastText (PCA 30 & 100)
- Freitexte per unsupervised **fastText** (skipgram, 100d) je Spalte eingebettet
- Spaltennamen-Embedding addiert (Positionskodierung), danach per-Vektor-LayerNorm
- PCA 30 bzw. 100 über den Embedding-Block, Rohtexte entfernt → `fast_text_pca{30,100}_fake_jobs.csv`

In [1]:
import pandas as pd
import numpy as np
import fasttext
from sklearn.decomposition import PCA

## Cleaned+Text laden & fastText trainieren
- Korpus = alle Freitextzellen (Zeilenumbrüche entfernt); unsupervised skipgram (dim=100)

In [2]:
df = pd.read_csv("../../data/preprocessed/cleaned_text_fake_jobs.csv")
text_cols = ["title", "company_profile", "description", "requirements", "benefits"]

corpus = pd.concat([df[c].fillna("") for c in text_cols]).str.replace(r"[\r\n]+", " ", regex=True)
with open("/tmp/ft_corpus.txt", "w") as f:
    f.write("\n".join(corpus.tolist()))

ft = fasttext.train_unsupervised("/tmp/ft_corpus.txt", model="skipgram", dim=100)
name_emb = {c: ft.get_sentence_vector(c) for c in text_cols}  # cached, constant per column

Read 6M words
Number of words:  49928
Number of labels: 0
Progress: 100.0% words/sec/thread:   67006 lr:  0.000000 avg.loss:  1.463131 ETA:   0h 0m 0s 45.4% words/sec/thread:   71439 lr:  0.027318 avg.loss:  1.351460 ETA:   0h 0m23s


## Zellen einbetten, Spaltennamen addieren, LayerNorm

In [3]:
parts = []
for c in text_cols:
    cells = df[c].fillna("").str.replace(r"[\r\n]+", " ", regex=True).tolist()
    e = np.vstack([ft.get_sentence_vector(t) for t in cells]) + name_emb[c]
    e = (e - e.mean(axis=1, keepdims=True)) / (e.std(axis=1, keepdims=True) + 1e-6)
    cols = [f"{c}_emb_{i}" for i in range(e.shape[1])]
    parts.append(pd.DataFrame(e, columns=cols, index=df.index))

emb = pd.concat(parts, axis=1)
base = df.drop(columns=text_cols)
print("embedding block", emb.shape)

embedding block (17880, 500)


## PCA 30 & 100, Rohtexte entfernt, speichern
- Erklärte Varianz wird je Variante geprintet

In [4]:
for n in (30, 100):
    pca = PCA(n_components=n, random_state=42)
    red = pca.fit_transform(emb.values)
    print(f"explained variance ({n} comps):", round(pca.explained_variance_ratio_.sum(), 4))
    out = pd.concat([base.reset_index(drop=True), pd.DataFrame(red, columns=[f"pca_{i}" for i in range(n)])], axis=1)
    out.to_csv(f"../../data/preprocessed/fast_text_pca{n}_fake_jobs.csv", index=False)
    print("saved", out.shape)

explained variance (30 comps): 0.6729
saved (17880, 45)
explained variance (100 comps): 0.853
saved (17880, 115)


## Verifikation

In [5]:
assert out.isna().sum().sum() == 0
print("text columns removed:", not any(c in out.columns for c in text_cols))

text columns removed: True
